# 03. ReAct 에이전트

## 학습 목표
- ReAct(Reasoning + Acting) 패턴의 개념을 이해한다
- `create_react_agent`로 에이전트를 1줄로 생성할 수 있다
- Tool을 정의하고 에이전트에 연결하여 다중 스텝 추론을 수행할 수 있다
- 시스템 프롬프트로 에이전트에 역할을 부여할 수 있다
- 대화 히스토리를 유지하는 에이전트를 구현할 수 있다

## 1. 환경 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env 파일에 설정되어 있는지 확인하세요!"

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5-mini")

print("환경 설정 완료!")

c:\Users\kimsa\Desktop\VS_ryo\cs_prac\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


환경 설정 완료!


## 2. Tool 3개 정의

에이전트가 사용할 도구를 `@tool` 데코레이터로 정의합니다.

In [4]:
from langchain_core.tools import tool
from datetime import datetime

@tool
def search_web(query: str) -> str:
    """웹에서 정보를 검색합니다. 최신 정보나 사실 확인이 필요할 때 사용합니다."""
    # 시뮬레이션 (실제로는 검색 API 호출)
    mock_results = {
        "LangGraph": "LangGraph는 LangChain 팀이 개발한 에이전트 오케스트레이션 프레임워크입니다. 상태 기반 그래프로 복잡한 AI 워크플로우를 구성할 수 있습니다.",
        "ReAct": "ReAct는 Reasoning과 Acting을 결합한 에이전트 패턴입니다. LLM이 생각(Thought) -> 행동(Action) -> 관찰(Observation) 사이클을 반복합니다.",
        "GPT-5": "GPT-5는 OpenAI의 최신 대규모 언어 모델입니다.",
    }
    for key, value in mock_results.items():
        if key.lower() in query.lower():
            return value
    return f"'{query}'에 대한 검색 결과: 관련 정보를 찾았습니다. AI 에이전트 기술이 빠르게 발전하고 있습니다."

@tool
def calculate(expression: str) -> str:
    """수학 계산을 수행합니다. Python 수식을 입력하세요."""
    try:
        # 안전한 수식 평가 (기본 연산만 허용)
        allowed_names = {"abs": abs, "round": round, "min": min, "max": max, "pow": pow}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {e}"

@tool
def get_current_time() -> str:
    """현재 날짜와 시간을 반환합니다."""
    now = datetime.now()
    return now.strftime("%Y년 %m월 %d일 %H시 %M분")

tools = [search_web, calculate, get_current_time]
print(f"정의된 Tool 목록: {[t.name for t in tools]}")

정의된 Tool 목록: ['search_web', 'calculate', 'get_current_time']


## 3. create_react_agent로 에이전트 생성 (1줄!)

`create_react_agent`는 LLM + Tool을 받아서 완전한 ReAct 에이전트 그래프를 자동으로 만들어줍니다.

In [5]:
from langgraph.prebuilt import create_react_agent

# 에이전트 생성 - 이것이 전부입니다!
agent = create_react_agent(llm, tools)

# 그래프 구조 확인
agent.get_graph().print_ascii()

        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+.          
          .         .         
        ..           ..       
       .               .      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


C:\Users\kimsa\AppData\Local\Temp\ipykernel_65632\781176319.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


## 4. 단일 질문 테스트

에이전트가 Tool을 적절히 선택하여 사용하는지 확인합니다.

In [6]:
from langchain_core.messages import HumanMessage

# 테스트 1: 계산 질문
print("=== 계산 질문 ===")
result = agent.invoke({
    "messages": [HumanMessage(content="1234 * 5678은 얼마인가요?")]
})
# 전체 메시지 흐름 출력
for msg in result["messages"]:
    if msg.type == "human":
        print(f"[사용자] {msg.content}")
    elif msg.type == "ai" and msg.content:
        print(f"[AI] {msg.content}")
    elif msg.type == "ai" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"[Tool 호출] {tc['name']}({tc['args']})")
    elif msg.type == "tool":
        print(f"[Tool 결과] {msg.content}")

print()

# 테스트 2: 시간 질문
print("=== 시간 질문 ===")
result = agent.invoke({
    "messages": [HumanMessage(content="지금 몇 시야?")]
})
print(f"최종 응답: {result['messages'][-1].content}")
print()

# 테스트 3: 검색 질문
print("=== 검색 질문 ===")
result = agent.invoke({
    "messages": [HumanMessage(content="ReAct 패턴이 뭔지 검색해서 알려줘")]
})
print(f"최종 응답: {result['messages'][-1].content}")

=== 계산 질문 ===
[사용자] 1234 * 5678은 얼마인가요?
[Tool 호출] calculate({'expression': '1234*5678'})
[Tool 결과] 1234*5678 = 7006652
[AI] 1234 × 5678 = 7,006,652

=== 시간 질문 ===
최종 응답: 지금 오후 2시 12분이야.

=== 검색 질문 ===
최종 응답: 요청하신 대로 웹에서 찾아 정리한 ReAct 패턴 설명(한국어)입니다.

요약
- ReAct은 "Reasoning + Acting"의 줄임말로, 언어 모델이 내부적 추론(Thought)과 외부 행동(Action)을 번갈아 수행하는 에이전트 패턴입니다.
- 모델은 생각(Thought)을 자연어로 서술하고, 필요한 경우 도구 호출(검색, 계산, 데이터베이스 질의 등)을 하는 Action을 실행합니다. 도구의 응답(Observation)을 받고 다시 생각을 이어가며 최종 답(Final Answer)에 도달합니다.

구성 요소(일반적인 포맷)
- Thought: 모델의 현재 사고(왜 그런 행동을 하는지, 다음 단계 계획 등)
- Action: 도구 호출(예: Search[질문], Calc[수식], Lookup[키])
- Observation: 도구로부터 받은 응답
- (반복)
- Final Answer: 사용자가 받을 최종 응답

간단한 예시
사용자 질문: "한국 인구는 얼마지?"
대화 예시:
- Thought: 최신 수치를 확인해야 한다.
- Action: Search["대한민국 인구 2024"]
- Observation: (검색 결과 -> 예: 통계청 추정치 ~5,100만)
- Thought: 검색 결과를 바탕으로 확인/요약한다.
- Final Answer: "통계청 추정에 따르면 대한민국 인구는 약 5,100만 명입니다 (출처: 통계청, 2024년 추정)."

장점
- 외부 도구(웹 검색, 계산기, 데이터베이스 등)를 자연스럽게 활용해 최신 정보나 계산을 정확히 반영할 수 있음.
- 모델의 사고 

## 5. 다중 스텝 추론 테스트

연구 보조 도구를 만들어 여러 단계의 추론이 필요한 질문을 테스트합니다.

In [7]:
# 연구용 Tool 추가
@tool
def search_papers(topic: str) -> str:
    """학술 논문을 검색합니다. 연구 주제를 입력하면 관련 논문 목록을 반환합니다."""
    papers = {
        "transformer": "1) Attention Is All You Need (Vaswani et al., 2017)\n2) BERT: Pre-training of Deep Bidirectional Transformers (Devlin et al., 2019)\n3) GPT-4 Technical Report (OpenAI, 2023)",
        "agent": "1) ReAct: Synergizing Reasoning and Acting (Yao et al., 2023)\n2) Toolformer: Language Models Can Teach Themselves to Use Tools (Schick et al., 2023)\n3) AutoGPT: An Autonomous GPT-4 Experiment (2023)",
        "rag": "1) Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (Lewis et al., 2020)\n2) Self-RAG: Learning to Retrieve, Generate, and Critique (Asai et al., 2023)",
    }
    for key, value in papers.items():
        if key in topic.lower():
            return value
    return f"'{topic}' 관련 논문: 다양한 연구가 진행 중입니다."

@tool
def summarize_text(text: str) -> str:
    """텍스트를 요약합니다. 긴 텍스트를 핵심 내용 위주로 요약합니다."""
    # 시뮬레이션 (실제로는 LLM 호출)
    words = text.split()
    if len(words) > 20:
        return "요약: " + " ".join(words[:20]) + "..."
    return f"요약: {text}"

@tool
def compare_items(item1: str, item2: str) -> str:
    """두 항목을 비교 분석합니다."""
    return f"비교 분석: '{item1}' vs '{item2}' - 두 항목 모두 AI 분야에서 중요한 개념이며, 각각 고유한 장점이 있습니다."

# 연구용 에이전트 생성
research_tools = [search_papers, summarize_text, compare_items, calculate]
research_agent = create_react_agent(llm, research_tools)

# 복합 질문 테스트
print("=== 다중 스텝 추론 ===")
result = research_agent.invoke({
    "messages": [HumanMessage(
        content="AI agent 관련 논문을 찾아보고, 어떤 연구들이 있는지 요약해줘"
    )]
})

# 전체 실행 과정 출력
for msg in result["messages"]:
    if msg.type == "human":
        print(f"\n[사용자] {msg.content}")
    elif msg.type == "ai" and msg.content:
        print(f"\n[AI 응답] {msg.content}")
    elif msg.type == "ai" and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"\n[Tool 호출] {tc['name']}({tc['args']})")
    elif msg.type == "tool":
        print(f"[Tool 결과] {msg.content[:100]}..." if len(msg.content) > 100 else f"[Tool 결과] {msg.content}")

C:\Users\kimsa\AppData\Local\Temp\ipykernel_65632\2986885625.py:31: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(llm, research_tools)


=== 다중 스텝 추론 ===

[사용자] AI agent 관련 논문을 찾아보고, 어떤 연구들이 있는지 요약해줘

[Tool 호출] search_papers({'topic': 'AI agents survey: agentic AI, autonomous agents, multi-agent systems, embodied agents, language-model agents'})
[Tool 결과] 1) ReAct: Synergizing Reasoning and Acting (Yao et al., 2023)
2) Toolformer: Language Models Can Tea...

[Tool 호출] compare_items({'item1': 'ReAct: Synergizing Reasoning and Acting (Yao et al., 2023)', 'item2': 'Toolformer: Language Models Can Teach Themselves to Use Tools (Schick et al., 2023)'})
[Tool 결과] 비교 분석: 'ReAct: Synergizing Reasoning and Acting (Yao et al., 2023)' vs 'Toolformer: Language Models ...

[Tool 호출] search_papers({'topic': 'Survey papers on autonomous agents, agentic AI, and language-model agents'})
[Tool 결과] 1) ReAct: Synergizing Reasoning and Acting (Yao et al., 2023)
2) Toolformer: Language Models Can Tea...

[AI 응답] 좋습니다 — AI agent(에이전트) 관련 주요 연구 주제들을 카테고리별로 정리하고, 대표 논문/작업들을 짧게 요약해드릴게요. 원하는 세부 분야(예: LLM 기반 에이전트, 로보틱스·임베디드 에이전트, 멀티에이전트 강화학습, 벤치마크

## 6. 시스템 프롬프트로 역할 부여

`create_react_agent`에 `prompt` 파라미터로 시스템 프롬프트를 전달할 수 있습니다.

In [8]:
from langchain_core.messages import SystemMessage

# 시스템 프롬프트로 역할 부여
system_prompt = """당신은 친절한 AI 수학 튜터입니다.
학생의 질문에 대해:
1. 먼저 개념을 쉽게 설명하고
2. calculate 도구로 예시 계산을 보여주고
3. 학생이 이해했는지 확인하는 질문을 던지세요
항상 한국어로 답변하세요."""

# 시스템 프롬프트가 포함된 에이전트
math_tutor = create_react_agent(
    llm,
    [calculate],
    prompt=system_prompt
)

# 테스트
result = math_tutor.invoke({
    "messages": [HumanMessage(content="피타고라스 정리가 뭐예요? 예시로 보여주세요.")]
})
print(result["messages"][-1].content)

C:\Users\kimsa\AppData\Local\Temp\ipykernel_65632\3549515514.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_tutor = create_react_agent(


개념 설명 (쉽게)
- 피타고라스 정리(피타고라스의 정리)는 직각삼각형에만 적용돼요.
- 직각삼각형에서 두 직각을 이루는 변들을 '밑변/높이' 또는 보통 a, b라 하고, 직각에 맞지 않는 가장 긴 변(빗변)을 c라 하면,
  a^2 + b^2 = c^2 가 성립합니다.
- 즉 두 짧은 변의 제곱의 합이 빗변의 제곱과 같다는 뜻이에요.

예시 계산 (calculate 도구로 확인한 계산)
- 예: a = 3, b = 4 일 때
  3^2 + 4^2 = 25
  c = sqrt(25) = 5

질문 (이해 확인)
- 이해했나요? 연습 문제 하나: 한 직각삼각형에서 두 변이 6과 8이라면 빗변의 길이는 얼마일까요? (한 번 풀어보세요 — 필요하면 풀이를 같이 보여드릴게요.)


## 7. 대화 히스토리 유지 (ResearchAssistant 클래스)

에이전트가 이전 대화를 기억하도록 히스토리를 관리하는 클래스를 만듭니다.

In [9]:
class ResearchAssistant:
    """대화 히스토리를 유지하는 연구 보조 에이전트"""
    
    def __init__(self):
        self.agent = create_react_agent(
            llm,
            [search_web, search_papers, calculate, get_current_time],
            prompt="당신은 연구 보조 AI입니다. 한국어로 답변하세요. 이전 대화 내용을 참고하여 일관성 있게 답변하세요."
        )
        self.history = []  # 대화 히스토리
    
    def chat(self, user_input: str) -> str:
        """사용자 입력에 대해 응답하고 히스토리를 유지합니다."""
        # 새 사용자 메시지 추가
        self.history.append(HumanMessage(content=user_input))
        
        # 에이전트 실행 (전체 히스토리 전달)
        result = self.agent.invoke({"messages": self.history})
        
        # AI 응답을 히스토리에 추가
        ai_message = result["messages"][-1]
        self.history.append(ai_message)
        
        return ai_message.content
    
    def show_history(self):
        """대화 히스토리를 출력합니다."""
        for msg in self.history:
            role = "사용자" if msg.type == "human" else "AI"
            print(f"[{role}] {msg.content[:200]}")
        print(f"\n총 {len(self.history)}개의 메시지")

# 사용 예시
assistant = ResearchAssistant()

# 첫 번째 대화
print("=== 대화 1 ===")
response = assistant.chat("LangGraph에 대해 검색해줘")
print(response)
print()

# 두 번째 대화 (이전 대화 참조)
print("=== 대화 2 ===")
response = assistant.chat("방금 알려준 내용을 한 줄로 요약해줘")
print(response)
print()

# 히스토리 확인
print("=== 대화 히스토리 ===")
assistant.show_history()

C:\Users\kimsa\AppData\Local\Temp\ipykernel_65632\1072405050.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  self.agent = create_react_agent(


=== 대화 1 ===
요청하신 LangGraph 관련 간단 검색 결과 및 요약입니다.

요약
- LangGraph는 LangChain 팀에서 만든 “에이전트 오케스트레이션(Agent orchestration)” 프레임워크입니다.
- 핵심 아이디어는 상태(state) 기반의 그래프 구조를 통해 복잡한 AI 워크플로우(여러 단계의 의사결정·작업 흐름)를 구성하고 제어하는 것이라는 점입니다.
- 이를 통해 여러 에이전트/툴 호출을 연결하고, 조건부 전환(분기), 상태 보존 및 디버깅/시각화 같은 기능을 제공하는 것으로 알려져 있습니다.

가능한 활용 사례
- 다단계 챗봇 워크플로우(질문 분해 → 외부 데이터 검색 → 응답 생성)
- 자동화 파이프라인(문서 처리, 요약, 검증)
- 복합 도구 조합(agent + retriever + tool 호출) 관리 및 오케스트레이션

LangChain과의 관계
- LangGraph는 LangChain 프로젝트 팀에서 나온 구성 요소/프레임워크로, LangChain의 에코시스템 내에서 “복잡한 에이전트 흐름을 그래프 형태로 설계·운영”하기 위한 도구로 이해하면 됩니다.
- LangChain이 LLM 래퍼, 체인(chain) 구성 등 넓은 기능을 제공한다면, LangGraph는 특히 상태 기반 그래프 오케스트레이션에 초점을 둡니다.

원하시면 다음을 찾아드릴게요
- 공식 문서(Documentation) 링크 및 GitHub 저장소
- 설치 방법 및 간단한 예제 코드(Python)
- LangGraph 관련 튜토리얼/블로그 포스트
- 학술·기술적 비교(예: 다른 오케스트레이션 도구와의 차이점)

어떤 것을 먼저 찾아드릴까요? (예: 공식 문서/깃허브 링크, 설치·코드 예제, 아니면 논문/블로그 자료)

=== 대화 2 ===
LangGraph는 LangChain 팀이 만든 상태 기반 그래프를 이용해 복잡한 에이전트·툴 워크플로우를 오케스트레이션하는 프레임워크입니다.

=== 대화 히스토리 ===
[사용자] LangGraph에 대해 

## 8. 실습 과제: 나만의 ReAct 에이전트 만들기

### 요구사항
1. 자신만의 Tool 3개 이상을 정의하세요 (예: 번역, 코드 실행, 날씨 등)
2. 시스템 프롬프트로 에이전트에 특정 역할을 부여하세요
3. `create_react_agent`로 에이전트를 생성하세요
4. 대화 히스토리를 유지하는 클래스로 감싸세요
5. 최소 3턴의 대화로 테스트하세요

In [11]:
# 1) Tool 정의 (3개 이상)
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent


@tool
def translate_line(text: str, target_language: str) -> str:
    """텍스트를 target_language(예: English, Japanese)로 번역한 것처럼 모의 응답을 반환합니다."""
    lang = (target_language or "English").strip()
    return f"[mock 번역 → {lang}] {text}"


@tool
def get_weather_demo(city: str) -> str:
    """도시 이름에 대한 가짜 날씨 문자열을 반환합니다(실습용)."""
    c = (city or "").strip() or "서울"
    return f"{c}: 맑음, 기온 21°C (데모 데이터)"


@tool
def run_math(expression: str) -> str:
    """간단한 수식을 계산합니다 (+, -, *, /, **, 괄호)."""
    expr = (expression or "").strip()
    if not expr:
        return "수식이 비었습니다."
    try:
        allowed = {"abs": abs, "round": round, "min": min, "max": max, "pow": pow}
        val = eval(expr, {"__builtins__": {}}, allowed)
        return f"{expr} = {val}"
    except Exception as e:
        return f"계산 오류: {e}"


@tool
def count_chars(text: str) -> str:
    """문자열의 글자 수(공백 포함)를 반환합니다."""
    t = text or ""
    return f"글자 수: {len(t)}"


my_tools = [translate_line, get_weather_demo, run_math, count_chars]

# 2) 시스템 프롬프트 — 역할 부여
MY_SYSTEM_PROMPT = """당신은 실습용 다목적 비서입니다.
번역·날씨(데모)·계산·글자 수 도구를 상황에 맞게 호출하고,
도구 결과를 바탕으로 한국어로 짧게 답하세요.
이전 대화를 기억하고 후속 질문에 이어서 답할 수 있습니다."""

# 3) 에이전트 생성 (위 셀에서 llm = ChatOpenAI(model=\"gpt-5-mini\") 가 정의되어 있어야 합니다)
my_agent = create_react_agent(llm, my_tools, prompt=MY_SYSTEM_PROMPT)


# 4) 대화 히스토리를 유지하는 클래스
class MyReActChat:
    """메시지 목록을 유지하며 같은 에이전트 그래프를 반복 호출합니다."""

    def __init__(self, agent):
        self._agent = agent
        self._messages: list = []

    def send(self, user_text: str) -> str:
        self._messages.append(HumanMessage(content=user_text))
        result = self._agent.invoke({"messages": self._messages})
        self._messages = list(result["messages"])
        last = self._messages[-1]
        content = getattr(last, "content", "")
        return content if isinstance(content, str) else str(content)

    def clear(self) -> None:
        self._messages.clear()


session = MyReActChat(my_agent)

# 5) 테스트 — 최소 3턴
turns = [
    "부산 날씨 알려줘. 날씨 도구 써줘.",
    "아까 어느 도시 얘기했어? 한 단어로.",
    "'Hello world' 글자 수 세어줘. count_chars 써줘.",
]

for i, q in enumerate(turns, 1):
    print(f"\n=== 턴 {i} ===")
    print(f"[사용자] {q}")
    reply = session.send(q)
    print(f"[비서] {reply}")


C:\Users\kimsa\AppData\Local\Temp\ipykernel_65632\1979015673.py:51: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  my_agent = create_react_agent(llm, my_tools, prompt=MY_SYSTEM_PROMPT)



=== 턴 1 ===
[사용자] 부산 날씨 알려줘. 날씨 도구 써줘.
[비서] 부산: 맑음, 기온 21°C (데모 데이터)

=== 턴 2 ===
[사용자] 아까 어느 도시 얘기했어? 한 단어로.
[비서] 부산

=== 턴 3 ===
[사용자] 'Hello world' 글자 수 세어줘. count_chars 써줘.
[비서] 글자 수: 11
